# 02 – Feature engineering
Builds the 20 predictors and the three targets from the saved price files, then runs the three leakage checks (correlation, truncation test and target alignment).

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

BASE     = "/content/drive/MyDrive/CS5500_dissertation"
DATA     = f"{BASE}/data"
FEATURES = f"{BASE}/features"
RESULTS  = f"{BASE}/results"

os.makedirs(FEATURES, exist_ok=True)

TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "SPY"]

print("Ready. Source data:", DATA)

## 2. The indicator functions
Written from scratch with trailing windows only, so no future data can leak into a feature.

In [ ]:
def rsi(series, window=14):
    """Relative Strength Index. Measures whether recent moves have been
    predominantly up or down, scaled to 0-100. Above 70 is conventionally
    read as overbought, below 30 as oversold."""
    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Wilder's smoothing, expressed as an exponentially weighted mean
    avg_gain = gain.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/window, min_periods=window, adjust=False).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def macd(series, fast=12, slow=26, signal=9):
    """Moving Average Convergence Divergence. The difference between a fast
    and a slow exponential moving average, plus its own smoothed signal line.
    The histogram (macd minus signal) is the usual momentum reading."""
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()

    macd_line   = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()

    return macd_line, signal_line, macd_line - signal_line


print("Indicator functions defined")

## 3. Build the feature matrix
20 predictors (returns, volatility, RSI, MACD, moving-average ratios, volume, range, day of week) plus the three targets.

In [ ]:
def build_features(df):
    """Turn a raw OHLCV dataframe into a feature matrix with targets.

    Every feature at row t uses only data from row t or earlier.
    Every target at row t refers to day t+1.
    """
    px  = df["Adj Close"].astype(float)
    vol = df["Volume"].astype(float)
    hi  = df["High"].astype(float)
    lo  = df["Low"].astype(float)

    ret = px.pct_change()
    out = pd.DataFrame(index=df.index)

    # --- Lagged returns ------------------------------------------------
    # ret_lag_1 is today's return, which is known at the close of today.
    for lag in [1, 2, 3, 5, 10, 20]:
        out[f"ret_lag_{lag}"] = ret.shift(lag - 1)

    # Cumulative returns over trailing windows
    for window in [5, 10, 20]:
        out[f"ret_cum_{window}"] = px.pct_change(window)

    # --- Realised volatility -------------------------------------------
    for window in [10, 20]:
        out[f"vol_{window}"] = ret.rolling(window).std()

    # --- Momentum indicators -------------------------------------------
    out["rsi_14"] = rsi(px, 14)

    macd_line, signal_line, histogram = macd(px)
    out["macd"]      = macd_line
    out["macd_hist"] = histogram

    # Distance from moving averages, expressed as a ratio so it is
    # comparable across assets with very different price levels
    for window in [20, 50]:
        out[f"px_over_ma_{window}"] = px / px.rolling(window).mean() - 1

    # --- Volume ---------------------------------------------------------
    out["vol_change"] = vol.pct_change()
    out["vol_ratio"]  = vol / vol.rolling(20).mean()

    # --- Intraday range --------------------------------------------------
    out["hl_range"] = (hi - lo) / px

    # --- Calendar ---------------------------------------------------------
    out["day_of_week"] = df.index.dayofweek

    # --- Targets (the only legitimate use of a negative shift) -----------
    out["target_return"]    = ret.shift(-1)
    out["target_direction"] = (ret.shift(-1) > 0).astype(int)
    out["target_price"]     = px.shift(-1)

    # Carried forward for later use: converting return forecasts back into
    # implied prices, and computing transaction costs in the backtest
    out["price_today"] = px

    return out


print("build_features defined")

In [ ]:
feature_sets = {}

for t in TICKERS:
    raw = pd.read_csv(f"{DATA}/{t}.csv", index_col=0, parse_dates=True)
    feats = build_features(raw)

    before = len(feats)

    # Drop rows where the long rolling windows have not yet filled, and the
    # final row where tomorrow does not exist. No imputation - filling these
    # would invent data.
    feats = feats.dropna()

    feats.to_csv(f"{FEATURES}/{t}_features.csv")
    feature_sets[t] = feats

    print(f"{t}: {before} -> {len(feats)} rows  ({before - len(feats)} dropped)  "
          f"{feats.index.min().date()} to {feats.index.max().date()}")

n_features = len([c for c in feature_sets[TICKERS[0]].columns
                  if not c.startswith('target') and c != 'price_today'])
print(f"\n{n_features} predictor columns per asset")

## 4. Inspect what was built

In [ ]:
example = feature_sets["AAPL"]

predictors = [c for c in example.columns
              if not c.startswith("target") and c != "price_today"]

print("Predictors:")
for i, c in enumerate(predictors, 1):
    print(f"  {i:2}. {c}")

print("\nSummary statistics (AAPL):\n")
print(example[predictors].describe().T[["mean", "std", "min", "max"]].round(4).to_string())

## 5. Leakage verification
Three checks: feature–target correlation, a truncation test and target alignment.

In [ ]:
print("CHECK 1 - correlation of each feature with tomorrow's return")
print("          (AAPL; anything above 0.15 is suspicious)\n")

corr = (example[predictors]
        .corrwith(example["target_return"])
        .sort_values(key=abs, ascending=False))

for name, value in corr.items():
    flag = "  <-- INVESTIGATE" if abs(value) > 0.15 else ""
    print(f"  {name:<20} {value:>8.4f}{flag}")

print(f"\nLargest absolute correlation: {corr.abs().max():.4f}")

In [ ]:
print("CHECK 2 - truncation test\n")

CUTOFF = "2020-06-30"

raw_full = pd.read_csv(f"{DATA}/AAPL.csv", index_col=0, parse_dates=True)

# What the features look like when the whole history is available
full_view = build_features(raw_full).loc[CUTOFF]

# What they would have looked like standing on the cutoff date itself,
# with the future genuinely unknown
raw_truncated = raw_full.loc[:CUTOFF]
past_view = build_features(raw_truncated).loc[CUTOFF]

mismatches = []

for col in predictors:
    a, b = full_view[col], past_view[col]
    if pd.isna(a) and pd.isna(b):
        continue
    if not np.isclose(a, b, rtol=1e-9, atol=1e-12):
        mismatches.append((col, a, b))

if mismatches:
    print("LOOKAHEAD BIAS DETECTED in:\n")
    for col, a, b in mismatches:
        print(f"  {col:<20} full={a:.8f}  truncated={b:.8f}")
else:
    print(f"PASS - all {len(predictors)} features identical whether computed")
    print(f"       from full history or from data up to {CUTOFF} only.")
    print("       No feature depends on future information.")

In [ ]:
print("CHECK 3 - target alignment\n")

px  = raw_full["Adj Close"].astype(float)
ret = px.pct_change()

sample = example.index[500:505]

print(f"{'date':<12} {'target_return':>14} {'actual next-day':>17} {'match':>7}")
for d in sample:
    pos = ret.index.get_loc(d)
    actual_next = ret.iloc[pos + 1]
    stored      = example.loc[d, "target_return"]
    ok = np.isclose(stored, actual_next)
    print(f"{str(d.date()):<12} {stored:>14.6f} {actual_next:>17.6f} {str(ok):>7}")

# Direction must be consistent with the return it was derived from
consistent = ((example["target_return"] > 0).astype(int)
              == example["target_direction"]).all()
print(f"\nDirection consistent with return on every row: {consistent}")

## 6. Feature correlation structure
Highly correlated predictors do not bias results, but they do make coefficient interpretation unreliable and can destabilise training.

In [ ]:
corr_matrix = example[predictors].corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_pairs = (upper.stack()
                   .sort_values(ascending=False)
                   .loc[lambda s: s > 0.8])

if len(high_pairs):
    print("Feature pairs correlated above 0.80:\n")
    for (a, b), v in high_pairs.items():
        print(f"  {a:<20} {b:<20} {v:.3f}")
else:
    print("No feature pair exceeds 0.80 correlation.")

## 7. Append to the results log

In [ ]:
today = datetime.now().strftime("%Y-%m-%d")

lines = []
lines.append("")
lines.append("---")
lines.append("")
lines.append(f"## {today} - Notebook 02: feature engineering")
lines.append("")
lines.append(f"**Features constructed:** {len(predictors)} predictors per asset")
for c in predictors:
    lines.append(f"- {c}")
lines.append("")
lines.append("**Targets:** target_direction (binary), target_return (continuous), "
             "target_price (next-day adjusted close, for the price-level protocol)")
lines.append("")
lines.append("**Rows retained after dropping incomplete rolling windows:**")
for t, f in feature_sets.items():
    lines.append(f"- {t}: {len(f)} rows")
lines.append("")
lines.append("**Leakage verification**")
lines.append(f"- Check 1: maximum absolute correlation between any feature and the "
             f"next-day return is {corr.abs().max():.4f}, consistent with weak "
             f"short-horizon predictability and showing no feature encodes the target.")
lines.append(f"- Check 2: truncation test passed. Features computed from the full "
             f"history are numerically identical to features computed from data up "
             f"to {CUTOFF} only, confirming no rolling window looks forward.")
lines.append("- Check 3: target_return on day t verified equal to the realised return "
             "on day t+1; target_direction consistent with target_return on every row.")
lines.append("")
lines.append("**Deliberately not done here:** no scaling, normalisation, or splitting. "
             "Scaling is performed inside each walk-forward fold using training-fold "
             "statistics only. Scaling at this stage would introduce exactly the "
             "preprocessing leakage this dissertation investigates.")
lines.append("")
lines.append("**Next:** Notebook 03 - expanding-window walk-forward splitter")

entry = "\n".join(lines)

with open(f"{BASE}/results_log.md", "a") as f:
    f.write(entry)

print(entry)